# RO-055 Hybrid A* and path smoothing: every number in the Note

The car, the parking lot and the planners live in `images/carkit.py` (Reeds-Shepp and Dubins paths in
`images/carpaths.py`). This notebook runs each planner and prints the numbers the Note quotes. Nothing here is random,
so there is no seed to set.

In [1]:
import sys, math
sys.path.insert(0, 'images')
import numpy as np
import carkit as ck
from carpaths import word, length, cusps
print('collision test margin', ck.MARGIN, 'm')
print('tightest circle', ck.RHO, 'm; wheelbase', ck.WHEELBASE, 'm; largest steering angle', round(math.degrees(ck.STEER_MAX), 2), 'deg')
print('start', ck.START, ' goal', (ck.GOAL[0], ck.GOAL[1], round(math.degrees(ck.GOAL[2]))))
print('start body free:', not ck.collides([ck.START])[0], ' goal clearance', round(ck.clearance(ck.GOAL), 3), 'm')

collision test margin 0.1 m
tightest circle 5.0 m; wheelbase 2.5 m; largest steering angle 26.57 deg
start (2.0, 3.0, 0.0)  goal (12.0, 23.0, -90)
start body free: True  goal clearance 0.25 m


## 1. The grid route and the car that follows it

In [2]:
G = ck.grid_astar()
P = np.array(G['path'], float); d = np.diff(P, axis=0); ang = np.degrees(np.arctan2(d[:, 1], d[:, 0]))
corners = [(tuple(P[i + 1].astype(int)), round(c)) for i, c in enumerate(np.diff(ang)) if abs(c) > 1e-9]
print('route length', round(G['cost'], 2), 'm through', len(P), 'cells;', G['expanded'], 'cells expanded')
print(len(corners), 'corners:', corners)
print('last move heads', ang[-1], 'deg; the goal faces', round(math.degrees(ck.GOAL[2])), 'deg')
TR, HIT = ck.follow_route(G['path'])
print('follower collides:', HIT, 'after', round((len(TR) - 1) * 0.05, 2), 'm, at', np.round(TR[-1], 2))
from shapely.geometry import Polygon, box
b = Polygon(ck.body_corners(TR[-1]))
print('hits the parked car', [o for o in ck.PARKED if b.intersects(box(o[0], o[2], o[1], o[3]))])

route length 33.31 m through 31 cells; 115 cells expanded
8 corners: [((np.int64(6), np.int64(7)), -45), ((np.int64(16), np.int64(7)), 90), ((np.int64(16), np.int64(12)), 45), ((np.int64(14), np.int64(14)), -45), ((np.int64(14), np.int64(15)), 45), ((np.int64(13), np.int64(16)), -45), ((np.int64(13), np.int64(21)), 45), ((np.int64(12), np.int64(22)), -45)]
last move heads 90.0 deg; the goal faces -90 deg
follower collides: True after 4.05 m, at [5.63 4.53 0.81]
hits the parked car [(6.1, 7.9, 7.5, 11.5)]


## 2. The six arcs of the first expansion

In [3]:
print('turn per arc', ck.STEP / ck.RHO, 'rad =', round(math.degrees(ck.STEP / ck.RHO), 2), 'deg; cell diagonal', round(math.sqrt(2), 3), 'm')
for dd, k in ck.PRIMS:
    tr = ck.arc(ck.START, dd, k); e = tuple(tr[-1]); hit = bool(ck.collides(tr).any())
    key = ck.key_of(e); g = ck.arc_cost(dd, k, None)
    rs, gr = ck.rs_length(e), ck.h_grid(e)
    print(('forward' if dd > 0 else 'reverse'), k, np.round(e[:2], 3), round(math.degrees(e[2]), 2), 'deg; cell', key[:2],
          'heading range', key[2], '(', key[2] * 5, 'deg ); blocked' if hit else f'; g {g}, RS {rs:.3f}, grid {gr:.3f}, f {g + max(rs, gr):.3f}')

turn per arc 0.3 rad = 17.19 deg; cell diagonal 1.414 m
forward L [3.478 3.223] 17.19 deg; cell (3, 3) heading range 3 ( 15 ; g 1.5, RS 26.882, grid 32.314, f 33.814
forward S [3.5 3. ] 0.0 deg; cell (4, 3) heading range 0 ( 0 ; g 1.5, RS 26.497, grid 31.314, f 32.814
forward R [3.478 2.777] -17.19 deg; cell (3, 3) heading range 69 ( 345 ; g 1.5, RS 25.811, grid 32.314, f 33.814
reverse L [0.522 3.223] -17.19 deg; cell (1, 3) heading range 69 ( 345 deg ); blocked
reverse S [0.5 3. ] 0.0 deg; cell (1, 3) heading range 0 ( 0 deg ); blocked
reverse R [0.522 2.777] 17.19 deg; cell (1, 3) heading range 3 ( 15 deg ); blocked


## 3. Why one pose per cell and heading range

In [4]:
level = [ck.START]
for k in range(1, 6):
    nxt = [tuple(ck.arc(p, dd, kk)[-1]) for p in level for dd, kk in ck.PRIMS]
    print(k, 'arcs:', len(nxt), 'arc ends,', len(set(ck.key_of(p) for p in nxt)), 'different cells and heading ranges')
    level = nxt
for a, b in (('L', 'R'), ('R', 'L'), ('S', 'S')):
    e = ck.arc(tuple(ck.arc(ck.START, 1, a)[-1]), 1, b)[-1]
    print(a, 'then', b, np.round(e[:2], 3), round(math.degrees(e[2]), 2), 'deg', ck.key_of(e))

1 arcs: 6 arc ends, 6 different cells and heading ranges
2 arcs: 36 arc ends, 19 different cells and heading ranges
3 arcs: 216 arc ends, 71 different cells and heading ranges
4 arcs: 1296 arc ends, 183 different cells and heading ranges
5 arcs: 7776 arc ends, 455 different cells and heading ranges
L then R [4.955 3.447] 0.0 deg (5, 3, 0)
R then L [4.955 2.553] 0.0 deg (5, 3, 0)
S then S [5. 3.] 0.0 deg (5, 3, 0)


## 4. The guesses of the cost left, and the search without the shot

In [5]:
print('at the start: straight line', round(ck.h_euclid(ck.START), 2), 'm; Reeds-Shepp', round(ck.rs_length(ck.START), 2),
      'm; grid distance', round(ck.h_grid(ck.START), 2), 'm')
NS = {}
for h in ('euclid', 'rs', 'grid', 'max'):
    r = ck.hybrid_astar(h, shot=False); NS[h] = r
    e = r['poses'][-1]
    print(h, r['expanded'], 'expansions; cost', round(r['cost'], 2), '; ends', round(math.hypot(e[0] - 12, e[1] - 23), 2), 'm and',
          round(math.degrees(abs(math.remainder(e[2] - ck.GOAL[2], 2 * math.pi)))), 'deg from the goal')

at the start: straight line 22.36 m; Reeds-Shepp 27.31 m; grid distance 33.31 m


euclid 2318 expansions; cost 50.0 ; ends 0.09 m and 5 deg from the goal


rs 2006 expansions; cost 50.0 ; ends 0.09 m and 5 deg from the goal


grid 2011 expansions; cost 50.0 ; ends 0.16 m and 5 deg from the goal


max 1949 expansions; cost 48.5 ; ends 0.54 m and 5 deg from the goal


## 5. With the Reeds-Shepp shot

In [6]:
R = ck.hybrid_astar('max', shot=True, trace=True)
attempts = [e for e in R['log'] if 'shot' in e]
print(R['expanded'], 'expansions;', len(attempts), 'shots tried,', sum(not e['shot']['ok'] for e in attempts), 'blocked')
print('arcs:', len(R['segments']), 'all forward:', all(s['d'] > 0 for s in R['segments']), '->', len(R['segments']) * ck.STEP, 'm')
print('shot from', np.round(R['shot_from'][:2], 2), round(math.degrees(R['shot_from'][2]), 1), 'deg:', word(R['shot']),
      [round(s, 3) for _, s in R['shot']], round(length(R['shot']), 3), 'm')
print('path', round(R['length'], 2), 'm; cost', round(R['cost'], 3), '; switches', R['cusps'])
print('cost of the arcs', len(R['segments']) * ck.STEP, '+ cost of the shot', round(ck.rs_cost(R['shot']), 3),
      '+ switch into the shot', ck.SWITCH if R['shot'][0][1] < 0 else 0.0)
print('smallest clearance of the body along the path', round(min(ck.clearance(p) for p in R['poses']), 3), 'm (test margin', ck.MARGIN, 'm)')
for h in ('euclid', 'rs', 'grid'):
    print('with the shot,', h, ck.hybrid_astar(h, shot=True)['expanded'], 'expansions')

312 expansions; 64 shots tried, 63 blocked
arcs: 20 all forward: True -> 30.0 m
shot from [ 9.4  14.67] 206.3 deg: R- S- L- [-5.817, -3.346, -0.255] 9.418 m
path 39.42 m; cost 50.835 ; switches 1
cost of the arcs 30.0 + cost of the shot 18.835 + switch into the shot 2.0
smallest clearance of the body along the path 0.162 m (test margin 0.1 m)
with the shot, euclid 256 expansions


with the shot, rs 361 expansions
with the shot, grid 156 expansions


## 6. Smoothing

In [7]:
X = P
Y1, IT1, _ = ck.smooth(X, alpha=0.5, beta=0.1)
k0, _ = ck.turning(X); k1, _ = ck.turning(Y1)
hits = lambda y: int(ck.collides(np.column_stack([y, ck.headings_along(y, np.ones(len(y)))]), margin=0).sum())
print('grid route: settles after', IT1, 'sweeps; tightest turn before', round(k0.max(), 3), 'per m, after', round(k1.max(), 3),
      'per m = a circle of', round(1 / k1.max(), 2), 'm; body hits at', hits(Y1), 'of', len(Y1), 'points; end heading',
      round(math.degrees(ck.headings_along(Y1, np.ones(len(Y1)))[-1]), 1), 'deg')
Y2, IT2, _ = ck.smooth(X, alpha=0.0, beta=0.1)
print('data weight 0: settles after', IT2, 'sweeps; body hits at', hits(Y2), 'points')
i = int(np.where((X == [16, 7]).all(1))[0][0])
y1 = X[i] + 0.5 * (X[i] - X[i]) + 0.1 * (X[i + 1] + X[i - 1] - 2 * X[i])
print('first sweep at the corner', X[i - 1], X[i], X[i + 1], '->', y1)

grid route: settles after 20 sweeps; tightest turn before 1.571 per m, after 1.453 per m = a circle of 0.69 m; body hits at 25 of 31 points; end heading 97.1 deg


data weight 0: settles after 11292 sweeps; body hits at 24 points
first sweep at the corner [15.  7.] [16.  7.] [16.  8.] -> [15.9  7.1]


In [8]:
SM, SN, RR = ck.smooth_path(R)
for k, ((y, dd), (x, _)) in enumerate(zip(SM, ck.pieces(R))):
    s0, s1 = ck.steering(x, dd), ck.steering(y, dd)
    print('piece', k, 'forward' if dd > 0 else 'reverse', len(x), 'points;', SN[k]['iters'], 'sweeps;', len(SN[k]['fixed']), 'held;',
          'steering movement', round(np.abs(np.diff(s0)).sum()), '->', round(np.abs(np.diff(s1)).sum()), 'deg; largest angle',
          round(np.abs(s1).max(), 2), 'deg; clearance', round(ck.dense_clearance(x, dd), 3), '->', round(ck.dense_clearance(y, dd), 3), 'm;',
          'length', round(np.hypot(*np.diff(x, axis=0).T).sum(), 2), '->', round(np.hypot(*np.diff(y, axis=0).T).sum(), 2))
print('reruns', RR)
x, dd = ck.pieces(R)[0]; held = SN[0]['fixed']
dist = np.concatenate([[0], np.cumsum(np.hypot(*np.diff(x, axis=0).T))])
runs, start = [], held[0]
for a, b in zip(held, held[1:] + [None]):
    if b != a + 1:
        runs.append((round(dist[start], 1), round(dist[a], 1))); start = b
print('held stretches, in m along the forward drive of', round(dist[-1], 2), 'm:', runs)

piece 0 forward 61 points; 49 sweeps; 35 held; steering movement 133 -> 100 deg; largest angle 26.57 deg; clearance 0.162 -> 0.189 m; length 29.99 -> 29.98
piece 1 reverse 21 points; 1 sweeps; 15 held; steering movement 34 -> 34 deg; largest angle 26.57 deg; clearance 0.227 -> 0.227 m; length 9.42 -> 9.42
reruns 16
held stretches, in m along the forward drive of 29.99 m: [(np.float64(0.0), np.float64(0.5)), (np.float64(12.0), np.float64(20.5)), (np.float64(23.0), np.float64(30.0))]
